# Study_Design_Merge — Interim Reference File

Merges green (SDTM Test Identity) and yellow (COSMoS BC/DSS) into a single
reference file for study design and SoA-to-CDISC mapping.

## Architecture

Two sheets, two levels of resolution:

| Sheet | Level | Primary key | Rows | Purpose |
|---|---|---|---|---|
| Test_Identity | Concept | TESTCD | ~5,800 | Mapping prompt lookup: what concept? |
| Measurement_Specs | Specification | DS_Code | ~1,100 | Specification selection: which variant? |

**Test_Identity** carries every TESTCD from SDTM CT (green baseline), enriched with
COSMoS summary columns where a Biomedical Concept exists. The mapping prompt resolves
SoA terms to TESTCDs against this sheet.

**Measurement_Specs** carries every Dataset Specialization from COSMoS. For TESTCDs
selected in the mapping step, the study designer looks up available specifications
here — specimen, method, scale, units, LOINC.

The link between sheets is TESTCD (and NCIt_Code for precision).

## Inputs

| File | Track | Content |
|---|---|---|
| `machine_actionable/SDTM_Test_Identity.xlsx` | sdtm-test-codes | 5,781 test codes with NCIt identity |
| `interim/COSMoS_BC_DSS.xlsx` | cosmos-bc-dss | 1,123 dataset specializations |

## Output

`interim/Study_Design_Merge.xlsx`

## Pipeline Position

```
sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx  ──┐
                                                               ├── this notebook
cosmos-bc-dss/interim/COSMoS_BC_DSS.xlsx  ────────────────────┘
                                                               │
                                        cosmos-bc-dss/interim/Study_Design_Merge.xlsx
```


## 1. Setup


In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('STUDY DESIGN MERGE — INTERIM REFERENCE FILE')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')
print()

BASE_DIR = Path('..')              # cosmos-bc-dss/
REPO_ROOT = BASE_DIR / '..'       # cdisc-for-ai/

GREEN_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
YELLOW_FILE = BASE_DIR / 'interim' / 'COSMoS_BC_DSS.xlsx'

OUTPUT_DIR = BASE_DIR / 'interim'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / 'Study_Design_Merge.xlsx'

for f, label in [(GREEN_FILE, 'Green'), (YELLOW_FILE, 'Yellow')]:
    if f.exists():
        print(f'  {label}: {f}')
    else:
        raise FileNotFoundError(f'{label} file not found: {f}')

print(f'  Output: {OUTPUT_FILE}')


## 2. Load Input Files


In [ ]:
green = pd.read_excel(GREEN_FILE, sheet_name='Test Codes', dtype=str).fillna('')
yellow = pd.read_excel(YELLOW_FILE, sheet_name='BC_DSS', dtype=str).fillna('')

print(f'Green: {len(green):,} rows x {len(green.columns)} columns')
print(f'Yellow: {len(yellow):,} rows x {len(yellow.columns)} columns')
print(f'Yellow row types: {yellow["Row_Type"].value_counts().to_dict()}')


## 3. Build Sheet 1: Test_Identity

One row per TESTCD. Green baseline enriched with COSMoS summary.

COSMoS enrichment per TESTCD:
- `Has_COSMoS` — whether a COSMoS BC exists for this test code
- `COSMoS_DSS_Count` — number of dataset specializations
- `COSMoS_BC_Name` — BC name(s) from COSMoS
- `COSMoS_Categories` — BC categories (for discovery/grouping)
- `COSMoS_Hierarchy` — hierarchy path (for panel/group context)
- `COSMoS_Domains` — domains that have DSSs for this BC


In [ ]:
# Extract COSMoS summary per TESTCD from DSS rows
dss = yellow[(yellow['Row_Type'] == 'DSS') & (yellow['TESTCD'] != '')].copy()

def cosmos_summary_for_testcd(group):
    """Summarize COSMoS BC/DSS info for one TESTCD."""
    # A TESTCD may map to multiple BCs (rare: HCG, LENGTH, MICROCY)
    bc_names = group['BC_Name'].unique()
    categories = group['Categories'].unique()
    hierarchies = group['Hierarchy_Path'].unique()
    domains = sorted(group['Domain'].unique())

    return pd.Series({
        'Has_COSMoS': 'Yes',
        'COSMoS_DSS_Count': str(len(group)),
        'COSMoS_BC_Name': '; '.join(n for n in bc_names if n),
        'COSMoS_Categories': '; '.join(c for c in categories if c),
        'COSMoS_Hierarchy': '; '.join(h for h in hierarchies if h),
        'COSMoS_Domains': '; '.join(domains),
    })

cosmos_summary = dss.groupby('TESTCD').apply(cosmos_summary_for_testcd).reset_index()
cosmos_summary = cosmos_summary.rename(columns={'TESTCD': 'TESTCD'})

print(f'COSMoS summary: {len(cosmos_summary):,} TESTCDs with DSS coverage')


In [ ]:
# Merge green baseline with COSMoS summary
test_codes = green.merge(
    cosmos_summary,
    on='TESTCD',
    how='left'
).fillna('')

# Fill Has_COSMoS for unmatched rows
test_codes.loc[test_codes['Has_COSMoS'] == '', 'Has_COSMoS'] = 'No'
test_codes.loc[test_codes['COSMoS_DSS_Count'] == '', 'COSMoS_DSS_Count'] = '0'

n_with = (test_codes['Has_COSMoS'] == 'Yes').sum()
n_without = (test_codes['Has_COSMoS'] == 'No').sum()
print(f'Test_Identity: {len(test_codes):,} rows')
print(f'  With COSMoS: {n_with:,} ({n_with/len(test_codes)*100:.1f}%)')
print(f'  Without COSMoS: {n_without:,} ({n_without/len(test_codes)*100:.1f}%)')


## 4. Build Sheet 2: Measurement_Specs

One row per Dataset Specialization. Carries the TESTCD for linking back to Sheet 1.
Includes only DSS rows (operational specifications), not hierarchy or BC-only rows.


In [ ]:
SPEC_COLS = [
    # Link to Sheet 1
    'TESTCD', 'TEST', 'NCIt_Code',
    # BC identity
    'COSMoS_BC_ID', 'BC_Name', 'BC_Type',
    # DSS identity
    'DS_Code', 'DS_Name', 'Domain', 'Domain_Class',
    # Measurement specification
    'Result_Scale', 'Specimen', 'Specimen_NCIt', 'Method', 'Method_NCIt',
    # Reference codes
    'LOINC_Code_Count', 'LOINC_Code', 'Allowed_Units', 'Standard_Unit',
    # Contextual
    'location', 'laterality', 'evaluator',
]

# All DSS rows, including those without TESTCD (46 rows in non-test-code domains)
specs = yellow[yellow['Row_Type'] == 'DSS'][SPEC_COLS].copy()

print(f'Measurement_Specs: {len(specs):,} rows')
print(f'  With TESTCD: {(specs["TESTCD"] != "").sum():,}')
print(f'  Without TESTCD: {(specs["TESTCD"] == "").sum():,}')
print(f'  Domains: {specs["Domain"].value_counts().head(10).to_dict()}')
print(f'  With specimen: {(specs["Specimen"] != "").sum():,}')
print(f'  With LOINC: {(specs["LOINC_Code"] != "").sum():,}')


## 5. Coverage Analysis

How much of the green file is enriched by COSMoS?


In [ ]:
print('=== Coverage ===')
print(f'Green TESTCDs:         {len(test_codes):,}')
print(f'  With COSMoS DSS:     {n_with:,} ({n_with/len(test_codes)*100:.1f}%)')
print(f'  Without COSMoS:      {n_without:,} ({n_without/len(test_codes)*100:.1f}%)')
print()

# Domain breakdown for COSMoS coverage
with_cosmos = test_codes[test_codes['Has_COSMoS'] == 'Yes']
print('COSMoS coverage by green SDTM_Domains:')
from collections import Counter
domain_coverage = Counter()
domain_total = Counter()
for _, row in test_codes.iterrows():
    for d in str(row['SDTM_Domains']).split('; '):
        if d.strip():
            domain_total[d.strip()] += 1
            if row['Has_COSMoS'] == 'Yes':
                domain_coverage[d.strip()] += 1

for dom in sorted(domain_total, key=domain_total.get, reverse=True)[:15]:
    cov = domain_coverage.get(dom, 0)
    tot = domain_total[dom]
    pct = cov / tot * 100 if tot else 0
    print(f'  {dom:6s}  {cov:4d} / {tot:4d}  ({pct:5.1f}%)')


## 6. Write Output


In [ ]:
# Column definitions for Sheet 1
TESTCODE_COLS = [
    # Green identity
    ('NCIt_Code',           12),
    ('TESTCD',              12),
    ('TEST',                38),
    ('SDTM_Domains',        18),
    ('NCIt_Preferred_Term', 45),
    ('NCIt_Synonyms',       55),
    ('NCIt_Definition',     60),
    ('UMLS_CUI',            14),
    ('NCIm_CUI',            14),
    # COSMoS summary
    ('Has_COSMoS',          10),
    ('COSMoS_DSS_Count',    10),
    ('COSMoS_BC_Name',      40),
    ('COSMoS_Categories',   30),
    ('COSMoS_Hierarchy',    40),
    ('COSMoS_Domains',      16),
]

# Column definitions for Sheet 2
SPEC_COL_WIDTHS = [
    ('TESTCD',              12),
    ('TEST',                38),
    ('NCIt_Code',           12),
    ('COSMoS_BC_ID',        14),
    ('BC_Name',             35),
    ('BC_Type',             10),
    ('DS_Code',             14),
    ('DS_Name',             38),
    ('Domain',              8),
    ('Domain_Class',        14),
    ('Result_Scale',        14),
    ('Specimen',            22),
    ('Specimen_NCIt',       12),
    ('Method',              18),
    ('Method_NCIt',         12),
    ('LOINC_Code_Count',    10),
    ('LOINC_Code',          18),
    ('Allowed_Units',       25),
    ('Standard_Unit',       14),
    ('location',            20),
    ('laterality',          14),
    ('evaluator',           14),
]

print('Column definitions ready.')


In [ ]:
# ── Styles — green/yellow colour scheme matching upstream notebooks ──
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')
THIN_BORDER = Border(bottom=Side(style='thin', color='D0D0D0'))

# Header fills — from upstream notebooks
GREEN_HEADER = PatternFill('solid', fgColor='548235')   # Enrich notebook: HDR_FILL
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')  # Flatten notebook: YELLOW_HEADER
README_HEADER = PatternFill('solid', fgColor='595959')  # Flatten notebook: README_HEADER
README_SECTION = PatternFill('solid', fgColor='D9D9D9') # Flatten notebook: README_SECTION

# Data fills
COSMOS_YES = PatternFill('solid', fgColor='FFFCE8')     # Flatten notebook: YELLOW_FILL
COSMOS_NO = PatternFill('solid', fgColor='F2F2F2')      # grey


def write_sheet(ws, df, col_defs, header_fill=None):
    """Write dataframe to worksheet with formatting."""
    col_names = [c[0] for c in col_defs]
    col_widths = {c[0]: c[1] for c in col_defs}
    fill = header_fill or GREEN_HEADER

    # Headers
    for ci, name in enumerate(col_names, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = fill
        cell.alignment = WRAP

    # Data
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(col_names, 1):
            val = row.get(name, '')
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP

    # Column widths
    for ci, (name, width) in enumerate(col_defs, 1):
        ws.column_dimensions[get_column_letter(ci)].width = width

    # Freeze and filter
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(col_names))}1'


print('Writer functions ready.')


In [ ]:
wb = Workbook()

# ── README ──
ws_readme = wb.active
ws_readme.title = 'README'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('Study Design Merge — Interim Reference File', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    (f'Green source: {GREEN_FILE.name}', readme_font),
    (f'Yellow source: {YELLOW_FILE.name}', readme_font),
    ('Notebook: Study_Design_Merge.ipynb', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Test_Identity — one row per TESTCD. Green identity + COSMoS summary.', readme_font),
    ('  The mapping prompt resolves SoA terms against this sheet.', readme_font),
    ('Measurement_Specs — one row per Dataset Specialization.', readme_font),
    ('  For selected TESTCDs, look up specimen/method/scale/units/LOINC here.', readme_font),
    ('', None),
    ('TEST_CODES COLUMNS (green identity)', section_font),
    ('NCIt_Code — NCIt C-code for the test concept', readme_font),
    ('TESTCD — SDTM short test code (submission value)', readme_font),
    ('TEST — SDTM full test name (submission value)', readme_font),
    ('SDTM_Domains — domain memberships (semicolon-separated)', readme_font),
    ('NCIt_Preferred_Term — preferred term from SDTM Terminology', readme_font),
    ('NCIt_Synonyms — alternative names from NCIt (semicolon-separated)', readme_font),
    ('NCIt_Definition — definition from NCIt Thesaurus', readme_font),
    ('UMLS_CUI — Standard UMLS CUI for crosslinking (SNOMED, MeSH, etc.)', readme_font),
    ('NCIm_CUI — NCI Metathesaurus CUI (NCI-internal)', readme_font),
    ('', None),
    ('TEST_CODES COLUMNS (COSMoS summary)', section_font),
    ('Has_COSMoS — Yes if a COSMoS BC with DSS exists for this TESTCD', readme_font),
    ('COSMoS_DSS_Count — number of dataset specializations available', readme_font),
    ('COSMoS_BC_Name — biomedical concept name(s) from COSMoS', readme_font),
    ('COSMoS_Categories — BC categories for discovery and grouping', readme_font),
    ('COSMoS_Hierarchy — hierarchy path for panel/group context', readme_font),
    ('COSMoS_Domains — domains with dataset specializations', readme_font),
    ('', None),
    ('MEASUREMENT_SPECS COLUMNS', section_font),
    ('TESTCD, TEST, NCIt_Code — link to Test_Identity sheet', readme_font),
    ('COSMoS_BC_ID — COSMoS internal BC identifier', readme_font),
    ('BC_Name — biomedical concept name', readme_font),
    ('DS_Code — dataset specialization mnemonic code', readme_font),
    ('DS_Name — dataset specialization name', readme_font),
    ('Domain, Domain_Class — SDTM domain and observation class', readme_font),
    ('Result_Scale — Quantitative / Qualitative / etc.', readme_font),
    ('Specimen, Method — resolved SDTM CT submission values', readme_font),
    ('LOINC_Code — LOINC code(s) where available', readme_font),
    ('Allowed_Units, Standard_Unit — unit specifications', readme_font),
    ('location, laterality, evaluator — contextual qualifiers (sparse)', readme_font),
    ('', None),
    ('COVERAGE', section_font),
    (f'Total TESTCDs: {len(test_codes):,}', readme_font),
    (f'With COSMoS DSS: {n_with:,} ({n_with/len(test_codes)*100:.1f}%)', readme_font),
    (f'Without COSMoS: {n_without:,} ({n_without/len(test_codes)*100:.1f}%)', readme_font),
    (f'Dataset Specializations: {len(specs):,}', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('Interim file. Column naming not yet finalized. Built with AI assistance.', readme_font),
    ('Not an official CDISC product. Sources: NCI EVS + COSMoS public exports.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_readme.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_readme.column_dimensions['A'].width = 100
print(f'README: {len(readme_lines)} lines')


In [ ]:
# ── Test_Identity sheet ──
ws_tc = wb.create_sheet('Test_Identity')
tc_col_names = [c[0] for c in TESTCODE_COLS]
write_sheet(ws_tc, test_codes[tc_col_names], TESTCODE_COLS, header_fill=GREEN_HEADER)

# Override COSMoS summary columns (last 6) with yellow headers
cosmos_start = len(TESTCODE_COLS) - 6 + 1  # 1-indexed
for ci in range(cosmos_start, len(TESTCODE_COLS) + 1):
    ws_tc.cell(row=1, column=ci).fill = YELLOW_HEADER

# Conditional fill on Has_COSMoS column
has_cosmos_ci = tc_col_names.index('Has_COSMoS') + 1
for ri in range(2, len(test_codes) + 2):
    val = ws_tc.cell(row=ri, column=has_cosmos_ci).value
    if val == 'Yes':
        ws_tc.cell(row=ri, column=has_cosmos_ci).fill = COSMOS_YES
    elif val == 'No':
        ws_tc.cell(row=ri, column=has_cosmos_ci).fill = COSMOS_NO

print(f'Test_Identity: {len(test_codes):,} rows')


In [ ]:
# ── Measurement_Specs sheet ──
ws_specs = wb.create_sheet('Measurement_Specs')
spec_col_names = [c[0] for c in SPEC_COL_WIDTHS]
write_sheet(ws_specs, specs[spec_col_names], SPEC_COL_WIDTHS, header_fill=YELLOW_HEADER)

# Link columns (TESTCD, TEST, NCIt_Code) get green headers — they're the bridge
for ci in range(1, 4):  # first 3 columns
    ws_specs.cell(row=1, column=ci).fill = GREEN_HEADER

print(f'Measurement_Specs: {len(specs):,} rows')


## 7. Save and Summary


In [ ]:
wb.save(OUTPUT_FILE)
print(f'Output: {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB')
print()
print('=== Summary ===')
print(f'Test_Identity sheet:         {len(test_codes):,} rows  (one per TESTCD)')
print(f'  With COSMoS:            {n_with:,}')
print(f'  Without COSMoS:         {n_without:,}')
print(f'Measurement_Specs sheet:  {len(specs):,} rows  (one per DSS)')
print(f'  With specimen:          {(specs["Specimen"] != "").sum():,}')
print(f'  With LOINC:             {(specs["LOINC_Code"] != "").sum():,}')
print()
print('Reference file ready for mapping prompt.')
